# 00 Python Unit Tests - Wstęp do testowania
_Kamil Bartocha_ | _wersja 2.0_

## Rozkład jazdy

1. 🔹 **Piramida testów i rodzaje testów** - unit, integration, e2e, SUT/DOC
2. 🔹 **Wzorzec AAA** - Arrange-Act-Assert
3. 🔹 **TDD - cykl Red-Green-Refactor**

🐍 Każda sekcja zawiera ćwiczenia.

## 1. 🔹 Piramida testów i rodzaje testów

Test (test jednostkowy/unit test) to fragment kodu sprawdzający,
czy inny fragment kodu działa poprawnie. Pisanie testów pozwala:
- wykrywać błędy wcześnie - zanim dotrą do produkcji,
- bezpiecznie refaktoryzować kod bez obawy o regresje,
- dokumentować oczekiwane zachowanie modułu.

Piramida testów (test pyramid) opisuje trzy poziomy testów
i zalecane proporcje między nimi:

| Poziom | Typ testu | Co sprawdza | Szybkość |
|--------|-----------|-------------|----------|
| 1 (dół) | jednostkowy (unit) | 1 funkcja / 1 klasa | bardzo szybki |
| 2 (środek) | integracyjny (integration) | kilka komponentów | średni |
| 3 (szczyt) | e2e (end-to-end) | cały system | wolny |

**Test jednostkowy (unit test)** testuje jedną, izolowaną jednostkę
kodu - typowo jedną funkcję lub metodę klasy. Nie korzysta
z bazy danych, sieci ani systemu plików.

**Test integracyjny (integration test)** sprawdza współpracę
kilku komponentów, np. serwisu z bazą danych.

**Test e2e (end-to-end test)** symuluje rzeczywistego użytkownika
korzystającego z całego systemu - np. test przez przeglądarkę.

> 💡 **Tip:** Piramida wyznacza proporcje - testy jednostkowe
> powinny stanowić ok. 70% wszystkich testów, integracyjne 20%,
> a e2e tylko 10%.

### SUT i DOC

W testach jednostkowych rozróżniamy dwa rodzaje elementów:

- **SUT (System Under Test)** - klasa lub funkcja, którą testujemy.
  To jest "bohater" testu - jego zachowanie weryfikujemy.
- **DOC (Depended-On Component)** - komponenty, od których SUT
  zależy (np. baza danych, serwis e-mail, zewnętrzne API).
  W testach jednostkowych zastępujemy DOC atrapami (mock).

Przykład: testując metodę `OrderService.place_order()`:
- SUT: `OrderService`
- DOC: `PaymentGateway`, `EmailSender`, `InventoryRepository`

Izolowanie SUT od DOC sprawia, że test jest szybki,
powtarzalny i niezależny od zewnętrznych systemów.

In [ ]:
# Trzy typy testow - przyklad konceptualny
import sqlite3

# --- Test jednostkowy (unit test) ---
# Testujemy TYLKO logike jednej funkcji, bez zewnetrznych zaleznosci.

def is_even(n):
    return n % 2 == 0

def test_is_even_unit():
    # Arrange
    n = 4
    # Act
    result = is_even(n)
    # Assert
    assert result is True
    print("PASS: test_is_even_unit")

test_is_even_unit()


# --- Test integracyjny (integration test) ---
# Testujemy wspolprace funkcji z baza danych.

def save_and_fetch_user(conn, username):
    conn.execute("INSERT INTO users (name) VALUES (?)", (username,))
    conn.commit()
    row = conn.execute(
        "SELECT name FROM users WHERE name=?", (username,)
    ).fetchone()
    return row[0]

def test_save_user_integration():
    conn = sqlite3.connect(":memory:")
    conn.execute("CREATE TABLE users (name TEXT)")
    result = save_and_fetch_user(conn, "alice")
    assert result == "alice"
    print("PASS: test_save_user_integration")
    conn.close()

test_save_user_integration()

---

### 🐍 Ćwiczenia - piramida testów i rodzaje testów

**Ćw. 4.** Przeczytaj ponizsze funkcje testowe i zdecyduj:
czy to test jednostkowy, integracyjny czy e2e?
Wpisz odpowiedz zastepujac `...` w komentarzu przy kazdej funkcji.

**Ćw. 7.** W przykladzie klasa `OrderProcessor` korzysta
z `PaymentService` i `Logger`. Uzupelnij komentarze:
co jest SUT, a co DOC?

**Ćw. 8. *(Trudniejsze)*** Sklasyfikuj testy z listy jako
`unit` / `integration` / `e2e`. Wpisz typ zastepujac `...`.

**Ćw. 10.** Projekt ma 200 testow automatycznych. Ile testow
kazdego rodzaju zaleca piramida? Wylicz liczby i uzasadnij.

**Ćw. 11.** Zidentyfikuj SUT i DOC dla klasy `ReportGenerator`
zaleznej od `UserRepository`, `CsvExporter` i `EmailService`.

**Ćw. 12. *(Trudniejsze)*** Dla kazdego testu z listy podaj:
typ testu i uzasadnienie dlaczego tak uwazasz.


In [ ]:
# Ćw. 4: unit, integration czy e2e?
import os
import sqlite3


# Funkcja A
def multiply(a, b):
    return a * b

def test_a():
    assert multiply(3, 4) == 12
    print("test_a passed")

test_a()
# typ testu: ...


# Funkcja B
def get_config_value(key):
    return os.environ.get(key, "default")

def test_b():
    os.environ["APP_ENV"] = "test"
    assert get_config_value("APP_ENV") == "test"
    print("test_b passed")

test_b()
# typ testu: ...


# Funkcja C
def count_records(conn, table):
    cursor = conn.execute(f"SELECT COUNT(*) FROM {table}")
    return cursor.fetchone()[0]

def test_c():
    conn = sqlite3.connect(":memory:")
    conn.execute("CREATE TABLE items (id INTEGER)")
    conn.execute("INSERT INTO items VALUES (1)")
    assert count_records(conn, "items") == 1
    print("test_c passed")
    conn.close()

test_c()
# typ testu: ...

In [ ]:
# Ćw. 7: SUT i DOC

class PaymentService:
    def charge(self, amount):
        pass  # bramka platnosci - zewnetrzny system

class Logger:
    def log(self, message):
        pass  # zapis do pliku/bazy - zewnetrzna zaleznosc

class OrderProcessor:
    def __init__(self, payment_service, logger):
        self.payment = payment_service
        self.logger = logger

    def process(self, order_amount):
        self.payment.charge(order_amount)
        self.logger.log(f"Order processed: {order_amount}")
        return "OK"

# Uzupelnij - co jest SUT, a co DOC w tym tescie?
# SUT:  ...
# DOC1: ...
# DOC2: ...

print("SUT: ", ...)  # nazwa klasy, ktora testujemy
print("DOC: ", ...)  # lista klas, od ktorych SUT zalezy

In [ ]:
# Ćw. 8: sklasyfikuj testy
# hint: unit = izolowana funkcja/klasa, integration = kilka komponentow,
#       e2e = caly system z perspektywy uzytkownika

test_cases = [
    # (opis testu,                         "unit"/"integration"/"e2e")
    ("test_add_two_numbers",               ...),
    ("test_user_login_via_browser",        ...),
    ("test_save_order_to_database",        ...),
    ("test_validate_email_format",         ...),
    ("test_full_checkout_flow",            ...),
    ("test_send_welcome_email_via_smtp",   ...),
    ("test_calculate_discount_percentage", ...),
    ("test_api_returns_404_for_missing",   ...),
]

for name, test_type in test_cases:
    print(f"{name:<45s} -> {test_type}")

In [ ]:
# Ćw. 10: proporcje piramidy testow
# Projekt ma 200 testow automatycznych.
# Oblicz ile testow kazdego rodzaju zaleca piramida (70 / 20 / 10 %)
# i uzasadnij krotkim komentarzem dlaczego takie proporcje.

total_tests = 200

unit_count        = ...  # 70% - najszybsze, najtansze, testuja logike
integration_count = ...  # 20% - sprawdzaja wspolprace komponentow
e2e_count         = ...  # 10% - wolne, drogie, testuja caly system

# Uzasadnienie (zastap ...) :
# unit:        ...
# integration: ...
# e2e:         ...

print(f"Testy jednostkowe (unit):       {unit_count}")
print(f"Testy integracyjne (integ.):    {integration_count}")
print(f"Testy end-to-end (e2e):         {e2e_count}")
print(f"Suma:                           {unit_count + integration_count + e2e_count}")


In [ ]:
# Ćw. 11: SUT i DOC dla ReportGenerator
# Klasa ReportGenerator generuje raporty CSV i wysyla je mailem.
# Korzysta z: UserRepository (baza danych), CsvExporter (zapis pliku),
#             EmailService (wysylka maila).
# Zidentyfikuj SUT i wszystkie DOC, a potem uzupelnij print().

class UserRepository:
    def get_all(self):
        pass  # zapytanie do bazy danych

class CsvExporter:
    def export(self, data, filepath):
        pass  # zapis do systemu plikow

class EmailService:
    def send(self, to, subject, attachment):
        pass  # wysylka przez SMTP

class ReportGenerator:
    def __init__(self, repo, exporter, email_service):
        self.repo = repo
        self.exporter = exporter
        self.email_service = email_service

    def generate_and_send(self, recipient):
        users = self.repo.get_all()
        self.exporter.export(users, "/tmp/report.csv")
        self.email_service.send(recipient, "Report", "/tmp/report.csv")
        return "sent"

# Uzupelnij:
# SUT:  ...  (co testujemy?)
# DOC1: ...  (zewnetrzna zaleznosc 1)
# DOC2: ...  (zewnetrzna zaleznosc 2)
# DOC3: ...  (zewnetrzna zaleznosc 3)

print("SUT:", ...)
print("DOC:", [..., ..., ...])


In [ ]:
# Ćw. 12 *(Trudniejsze)*: typ testu + uzasadnienie
# Dla kazdego opisu podaj typ ("unit" / "integration" / "e2e")
# ORAZ krotkie uzasadnienie (jedna fraza) dlaczego tak uwazasz.
# hint: kluczowe pytania -
#   czy test ma zewnetrzne zaleznosci (baza, siec, plik)?
#   czy symuluje zachowanie uzytkownika w calym systemie?

cases = [
    # (opis,                                           typ,   uzasadnienie)
    ("sprawdza czy format_price(9.9) zwraca '9.90 zł'", ...,  ...),
    ("loguje sie przez przegladarke i sprawdza dashboard", ..., ...),
    ("zapisuje uzytkownika do bazy i odczytuje go",     ...,  ...),
    ("weryfikuje czy klasa Cart poprawnie liczy rabat",  ...,  ...),
    ("uruchamia caly flow zakupowy od koszyka do maila", ...,  ...),
    ("sprawdza wspolprace AuthService z TokenRepository", ..., ...),
]

for desc, test_type, reason in cases:
    print(f"Opis:          {desc}")
    print(f"Typ:           {test_type}")
    print(f"Uzasadnienie:  {reason}")
    print()


## 2. 🔹 Wzorzec AAA (Arrange-Act-Assert)

Wzorzec AAA (Arrange-Act-Assert) to konwencja struktury testu
jednostkowego, ktora dzieli test na trzy wyrazne fazy:

1. **Arrange (przygotowanie)** - tworzymy obiekty, inicjalizujemy
   dane wejsciowe i konfigurujemy potrzebne zaleznosci.
2. **Act (dzialanie)** - wywolujemy testowana funkcje lub metode
   dokladnie raz.
3. **Assert (weryfikacja)** - sprawdzamy czy wynik jest zgodny
   z oczekiwaniem.

Kazdy test powinien sprawdzac dokladnie jeden przypadek.
Dzieki temu test jest czytelny i latwy do debugowania.

```python
def test_something():
    # Arrange
    sut = SomeClass()
    # Act
    result = sut.some_method(input_value)
    # Assert
    assert result == expected_value
```

> 💡 **Tip:** Jezeli w sekcji Act pojawia sie wiecej niz jedno
> wywolanie, zazwyczaj oznacza to, ze test sprawdza za duzo naraz.
> Podziel go na kilka mniejszych testow.

In [ ]:
# Wzorzec AAA - przyklady

class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive")
        self.balance += amount

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("Insufficient funds")
        self.balance -= amount


# Przyklad 1: depozyt zwieksza saldo
def test_deposit_increases_balance():
    # Arrange
    account = BankAccount(balance=100)

    # Act
    account.deposit(50)

    # Assert
    assert account.balance == 150
    print("PASS: test_deposit_increases_balance")


# Przyklad 2: wyplata przekraczajaca saldo rzuca wyjatek
def test_withdraw_raises_when_insufficient_funds():
    # Arrange
    account = BankAccount(balance=50)

    # Act & Assert
    try:
        account.withdraw(200)
        print("FAIL: powinien rzucic ValueError")
    except ValueError as e:
        print(f"PASS: test_withdraw_raises_when_insufficient_funds ({e})")


test_deposit_increases_balance()
test_withdraw_raises_when_insufficient_funds()

---

### 🐍 Ćwiczenia - wzorzec AAA

**Ćw. 1.** Przeczytaj ponizszy test i oznacz kazda sekcje,
zastepujac `...` slowem `Arrange`, `Act` lub `Assert`.

**Ćw. 2.** Napisz reczny test dla funkcji `is_palindrome()`.
Uzyj wzorca AAA i sprawdz co najmniej dwa rózne przypadki.

**Ćw. 3. *(Trudniejsze)*** Wypisz liste przypadkow testowych
dla funkcji `divide(a, b)`. Dla kazdego podaj nazwe testu.
hint: pomysl o happy path, edge cases i bledzie (wyjatku).

In [ ]:
# Ćw. 1: oznacz sekcje AAA (zastap ... slowem Arrange / Act / Assert)

class Stack:
    def __init__(self):
        self._items = []

    def push(self, item):
        self._items.append(item)

    def pop(self):
        return self._items.pop()

    def is_empty(self):
        return len(self._items) == 0


def test_pop_returns_last_pushed_item():
    # --- ... ---
    stack = Stack()
    stack.push("a")
    stack.push("b")

    # --- ... ---
    result = stack.pop()

    # --- ... ---
    assert result == "b"
    print(f"result = {result!r}")


test_pop_returns_last_pushed_item()

In [ ]:
# Ćw. 2: reczny test dla is_palindrome()

def is_palindrome(text):
    return text == text[::-1]


def test_palindrome_returns_true_for_racecar():
    # Arrange
    ...
    # Act
    ...
    # Assert
    ...
    print("PASS: test_palindrome_returns_true_for_racecar")


def test_palindrome_returns_false_for_hello():
    # Arrange
    ...
    # Act
    ...
    # Assert
    ...
    print("PASS: test_palindrome_returns_false_for_hello")


test_palindrome_returns_true_for_racecar()
test_palindrome_returns_false_for_hello()

In [ ]:
# Ćw. 3: przypadki testowe dla divide(a, b)
# hint: pomysl o: dzielenie pozytywnych, ujemnych, przez 1,
#       przez 0, wynik float, argumenty blednego typu

def divide(a, b):
    if b == 0:
        raise ZeroDivisionError("Cannot divide by zero")
    return a / b


# Wypisz co najmniej 6 przypadkow testowych (nazwy funkcji testujacych):
test_cases = [
    "test_divide_10_by_2_returns_5",
    ...,  # happy path - inne liczby
    ...,  # dzielenie przez 1
    ...,  # liczby ujemne
    ...,  # wyjatek ZeroDivisionError
    ...,  # edge case - twoj pomysl
]

for case in test_cases:
    print(f"- {case}")

## 3. 🔹 TDD - cykl Red-Green-Refactor

TDD (Test-Driven Development / programowanie sterowane testami)
to technika, w ktorej piszemy test PRZED napisaniem kodu
produkcyjnego. Cykl TDD sklada sie z trzech krokow:

1. **Red (czerwony)** - piszemy test, ktory nie przechodzi,
   bo testowanego kodu jeszcze nie ma.
2. **Green (zielony)** - piszemy MINIMUM kodu, zeby test przeszedl.
   Nie martwimy sie o elegancje - liczy sie tylko zielony wynik.
3. **Refactor (refaktoryzacja)** - poprawiamy kod bez zmiany
   jego zachowania. Testy nadal musza przechodzic.

Korzysci TDD:
- wymusza myslenie o interfejsie zanim napiszemy implementacje,
- kazda funkcjonalnosc ma testy od samego poczatku,
- prowadzi do mniejszych, testowalnych funkcji.

| Krok | Kolor | Akcja |
|------|-------|-------|
| 1 | 🔴 Red | napisz test - musi failowac |
| 2 | 🟢 Green | napisz minimum kodu - test musi przejsc |
| 3 | 🔵 Refactor | ulepsz kod - testy nadal zielone |

> 💡 **Tip:** "Minimum kodu" oznacza dosłowne minimum.
> Mozna zwrocic stala wartosc jezeli test to akceptuje.
> Kolejny test zmusi nas do implementacji ogolnego rozwiazania.

In [ ]:
# TDD krok po kroku - przyklad FizzBuzz

# === KROK 1: RED ===
# Zaczynamy od testu. Brak implementacji -> NameError.
# def test_fizzbuzz_3_returns_fizz():
#     assert fizzbuzz(3) == "Fizz"  # NameError: fizzbuzz not defined

# === KROK 2: GREEN - minimum implementacji dla pierwszego testu ===
def fizzbuzz(n):
    if n % 15 == 0:
        return "FizzBuzz"
    if n % 3 == 0:
        return "Fizz"
    if n % 5 == 0:
        return "Buzz"
    return str(n)


# === KROK 3: REFACTOR - testy weryfikuja poprawnosc po zmianach ===
def test_fizzbuzz():
    assert fizzbuzz(1)  == "1"
    assert fizzbuzz(3)  == "Fizz"
    assert fizzbuzz(5)  == "Buzz"
    assert fizzbuzz(15) == "FizzBuzz"
    assert fizzbuzz(9)  == "Fizz"
    assert fizzbuzz(10) == "Buzz"
    print("PASS: test_fizzbuzz (Red -> Green -> Refactor)")


test_fizzbuzz()

---

### 🐍 Ćwiczenia - TDD

**Ćw. 5.** Dla klasy `BankAccount` wypisz minimum 5 przypadkow
testowych. Dla kazdego podaj: nazwe testu, wejscie i oczekiwany
wynik lub wyjatek.

**Ćw. 6. *(Trudniejsze)*** Napisz pseudokod TDD dla funkcji
`validate_email(email)`. Zacznij od testu (Red), potem dodaj
minimalną implementację (Green) i uruchom testy.

**Ćw. 9. *(Trudniejsze)*** Zaprojektuj strategie testow dla
klasy `ShoppingCart`. Dla kazdej metody wypisz co najmniej
dwa przypadki testowe (happy path + edge case / wyjatek).

In [ ]:
# Ćw. 5: przypadki testowe dla BankAccount

class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive")
        self.balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive")
        if amount > self.balance:
            raise ValueError("Insufficient funds")
        self.balance -= amount

    def get_balance(self):
        return self.balance


# Wypisz minimum 5 przypadkow testowych:
# format: ("nazwa_testu", "wejscie", "oczekiwany wynik lub wyjatek")
test_cases = [
    # przyklad:
    ("test_deposit_increases_balance", "balance=0, deposit(100)", "balance==100"),
    ...,  # depozyt - kolejny przypadek
    ...,  # wyplata
    ...,  # wyplata przekraczajaca saldo -> wyjatek
    ...,  # inicjalizacja z domyslnym saldem
    ...,  # twoj pomysl - edge case
]

for name, inputs, expected in test_cases:
    print(f"- {name}")
    print(f"    wejscie:   {inputs}")
    print(f"    oczekiwany: {expected}")

In [ ]:
# Ćw. 6: TDD - pseudokod i implementacja validate_email()
# hint: poprawny email zawiera '@' i '.' po '@'
#       przykladowe testy: 'user@example.com' -> True,
#       'no-at-sign' -> False, 'missing@dot' -> False

# KROK 1 - RED: napisz testy (pseudokod jako komentarze)
# def test_validate_email_valid_address():
#     ...

# def test_validate_email_missing_at():
#     ...

# def test_validate_email_missing_dot_after_at():
#     ...

# KROK 2 - GREEN: minimalna implementacja
def validate_email(email):
    ...


# KROK 3 - REFACTOR: napisz wlasciwe testy i uruchom
def test_validate_email_valid():
    assert validate_email("user@example.com") is True
    print("PASS: test_validate_email_valid")


def test_validate_email_invalid():
    assert validate_email("no-at-sign") is False
    print("PASS: test_validate_email_invalid")


test_validate_email_valid()
test_validate_email_invalid()

In [ ]:
# Ćw. 9: strategia testow dla ShoppingCart
# hint: dla kazdej metody pomysl o happy path i edge case / wyjatku

class ShoppingCart:
    def __init__(self):
        self.items = []

    def add_item(self, name, price, qty=1):
        if price < 0:
            raise ValueError("Price cannot be negative")
        self.items.append({"name": name, "price": price, "qty": qty})

    def remove_item(self, name):
        self.items = [i for i in self.items if i["name"] != name]

    def total(self):
        return sum(i["price"] * i["qty"] for i in self.items)

    def apply_discount(self, percent):
        if not 0 < percent <= 100:
            raise ValueError("Discount must be between 1 and 100")
        return self.total() * (1 - percent / 100)


# Zaprojektuj strategie - wypisz przypadki dla kazdej metody:
strategy = {
    "add_item": [
        ...,  # happy path
        ...,  # edge case lub wyjatek
    ],
    "remove_item": [
        ...,
        ...,
    ],
    "total": [
        ...,
        ...,
    ],
    "apply_discount": [
        ...,
        ...,
    ],
}

for method, cases in strategy.items():
    print(f"\n{method}:")
    for case in cases:
        print(f"  - {case}")